# UDA-DSN pipeline for Hindi → Sanskrit ASR

Implementation of the **Domain Separation Network (DSN)** unsupervised-domain-adaptation pipeline from *Anoop C. S., Prathosh A. P. & Ramakrishnan, "Unsupervised Domain Adaptation Schemes for Building ASR in Low-Resource Languages"* (ASRU 2021), Sec 2.2 & 3.4.

Source domain = **Hindi** (labeled). Target domain = **Sanskrit** (unlabeled). The DSN learns per-domain **private** encoders and a **shared** encoder whose features are domain-invariant (adversarially, via a gradient reversal layer) yet senone-discriminative; a shared decoder reconstructs the input from private+shared features, and a difference loss keeps the two subspaces orthogonal.

**Objective (Eq. 1):** `L = L_class + β·L_sim + γ·L_diff + δ·L_recon`, with `β=0.25`, `γ=0.075`, `δ=0.1` (Sec 3.4).

**Architecture (Sec 3.4):** input 1320-dim (40×3×11 filterbank+Δ+ΔΔ, spliced ±5); private encoders 4×512; shared encoder 6×1024; senone classifier 2×1024→3080; domain classifier 1×256→2; shared decoder 3 hidden→1320; every hidden layer BatchNorm+ReLU.

**Training (Sec 3.3–3.4):** SGD+momentum, lr 0.01 scaled ×0.95 every 20,000 steps, 20 epochs, batch 32; domain adversarial (similarity) loss activated only after 10,000 steps; the GRL factor α ramped 0→1 (Ganin & Lempitsky).

> **Runtime note:** hand-checked against the paper and the reference DSN (Bousmalis 2016) implementation, and smoke-tested end-to-end on real Kaldi-extracted features + senone labels. The dataset is prepared by the scripts in `prepare_data/` and wired in §10 (`num_senones=2472` from our Kaldi run).

## 1) Imports & device

In [ ]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2) Gradient Reversal Layer (GRL)

Inside the DSN this drives the **similarity loss** `L_sim` (Sec 2.2): identity on the forward pass, gradient negated and scaled by `α` on the backward pass. It makes the shared encoder push to *maximize* domain-classification loss (domain-invariant features) while the domain classifier still *minimizes* it. `α` ramps 0→1 with Ganin & Lempitsky's schedule (Sec 3.3, ref [8]).

In [ ]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


def grad_reverse(x, alpha=1.0):
    return GradReverse.apply(x, alpha)


def grl_alpha_schedule(progress):
    """Ganin & Lempitsky (2015) schedule (paper Sec 3.3, ref [8]).
    progress: fraction of total training steps completed, in [0, 1]."""
    return 2.0 / (1.0 + math.exp(-10.0 * progress)) - 1.0

## 3) DSN building blocks

Dimensions follow Sec 3.4 exactly. Every hidden layer is `Linear → BatchNorm1d → ReLU`. NLL over log-softmax is used for both classifiers; `nn.CrossEntropyLoss` (log-softmax + NLL, identical) is used throughout.

In [ ]:
def _mlp_stack(in_dim, hidden_dim, num_layers):
    layers = []
    for i in range(num_layers):
        d_in = in_dim if i == 0 else hidden_dim
        layers += [nn.Linear(d_in, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU()]
    return nn.Sequential(*layers)


class PrivateEncoder(nn.Module):
    """E_p: 4 hidden layers, 512 nodes (one per domain)."""
    def __init__(self, input_dim=1320, hidden_dim=512, num_layers=4):
        super().__init__()
        self.net = _mlp_stack(input_dim, hidden_dim, num_layers)

    def forward(self, x):
        return self.net(x)


class SharedEncoder(nn.Module):
    """E_c: 6 hidden layers, 1024 nodes (common to both domains)."""
    def __init__(self, input_dim=1320, hidden_dim=1024, num_layers=6):
        super().__init__()
        self.net = _mlp_stack(input_dim, hidden_dim, num_layers)

    def forward(self, x):
        return self.net(x)


class SenoneClassifier(nn.Module):
    """G: 2 hidden layers x 1024, output = num_senones (3080 for Hindi)."""
    def __init__(self, input_dim=1024, hidden_dim=1024, num_senones=3080):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, num_senones),
        )

    def forward(self, x):
        return self.net(x)


class DomainClassifier(nn.Module):
    """Z: 1 hidden layer x 256, output = 2 (source=0 / target=1)."""
    def __init__(self, input_dim=1024, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x):
        return self.net(x)


class SharedDecoder(nn.Module):
    """D: 3 hidden layers + output layer (1320).

    Paper Eq. (3) writes the decoder input as E_c(x) + E_p(x), but Sec 3.4 also
    states shared=1024 and private=512 dims, which cannot be summed elementwise.
    We honor the stated layer sizes and **concatenate** (1024+512=1536). To follow
    the '+' literally instead, set private_dim=1024 and sum in the DSN wrapper."""
    def __init__(self, input_dim, hidden_dim=1024, output_dim=1320, num_layers=3):
        super().__init__()
        layers = []
        d_in = input_dim
        for _ in range(num_layers):
            layers += [nn.Linear(d_in, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU()]
            d_in = hidden_dim
        layers += [nn.Linear(hidden_dim, output_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

## 4) DSN model

- `mode='train'` returns everything the four loss terms need. The domain classifier is fed the shared features **through the GRL**. Senone logits are computed only for the source domain (`L_class` is source-only, Eq. 1 / Sec 3.3: "Only θf and θd are updated during training with unlabelled Sanskrit").
- `mode='inference'` uses only the shared encoder → senone classifier path (Sec 3.5: the domain classifier, private encoders and decoder are ignored).

In [ ]:
class DSN(nn.Module):
    def __init__(self, input_dim=1320, shared_dim=1024, private_dim=512, num_senones=3080):
        super().__init__()
        self.private_source = PrivateEncoder(input_dim, private_dim)
        self.private_target = PrivateEncoder(input_dim, private_dim)
        self.shared_encoder = SharedEncoder(input_dim, shared_dim)
        self.senone_classifier = SenoneClassifier(shared_dim, shared_dim, num_senones)
        self.domain_classifier = DomainClassifier(shared_dim, 256)
        self.shared_decoder = SharedDecoder(shared_dim + private_dim, shared_dim, input_dim)

    def forward(self, x, domain='source', mode='train', alpha=1.0):
        private_encoder = self.private_source if domain == 'source' else self.private_target
        private_feat = private_encoder(x)
        shared_feat = self.shared_encoder(x)

        if mode == 'inference':
            return {"shared": shared_feat, "senone_logits": self.senone_classifier(shared_feat)}

        recon = self.shared_decoder(torch.cat([shared_feat, private_feat], dim=1))
        domain_logits = self.domain_classifier(grad_reverse(shared_feat, alpha))
        senone_logits = self.senone_classifier(shared_feat) if domain == 'source' else None
        return {
            "private": private_feat,
            "shared": shared_feat,
            "recon": recon,
            "senone_logits": senone_logits,
            "domain_logits": domain_logits,
        }

## 5) Loss functions

- **`difference_loss`** (Eq. 2) — the canonical Bousmalis (2016) form the paper cites: mean-center each feature dim, L2-normalize each row, then the mean of the squared shared/private cross-correlation matrix. Soft subspace orthogonality between shared and private components.
- **`reconstruction_loss`** (Eq. 3) — squared L2 error per frame, averaged over frames (MSE).
- **`simse_loss`** (Eq. 4) — scale-invariant MSE variant used in the loss-ablation (Sec 4.1.3).

All operate on already-flattened, **padding-free** frame tensors (the training loop drops padding before the forward pass, §6), so no masking is needed here.

In [ ]:
def difference_loss(private_feat, shared_feat):
    """Eq. (2): mean-centered, row-normalized, mean of squared cross-correlation."""
    if private_feat.size(0) == 0:
        return private_feat.new_zeros(())
    private_feat = private_feat - private_feat.mean(dim=0, keepdim=True)
    shared_feat = shared_feat - shared_feat.mean(dim=0, keepdim=True)
    private_feat = F.normalize(private_feat, dim=1)
    shared_feat = F.normalize(shared_feat, dim=1)
    correlation = private_feat.t() @ shared_feat          # [Dp, Ds]
    return (correlation ** 2).mean()


def reconstruction_loss(x, x_hat):
    """Eq. (3): squared L2 per frame, meaned over frames (MSE)."""
    if x.size(0) == 0:
        return x.new_zeros(())
    return ((x - x_hat) ** 2).sum(dim=1).mean()


def simse_loss(x, x_hat):
    """Eq. (4): scale-invariant MSE. k = feature dim."""
    if x.size(0) == 0:
        return x.new_zeros(())
    diff = x - x_hat
    k = diff.size(1)
    term1 = (diff ** 2).sum(dim=1) / k
    term2 = (diff.sum(dim=1) ** 2) / (k ** 2)
    return (term1 - term2).mean()

## 6) Data loading

Lazy one-`.npy`-per-utterance loader (realistic for Kaldi output). Each item carries its **utterance ID** so predictions can be matched back to references; `collate_fn` pads to the batch max length and returns a boolean **mask** of real vs. padded frames.

> **Frame-level vs. utterance batching (Sec 3.3):** the paper's "batch size of 32" and "the same number of frames from source and target at every epoch" are stated at the **frame** level — the acoustic model is a per-frame feed-forward DNN with no recurrence. Utterance batching (below) is a convenience; with it, `batch_size=32` means ~32 utterances × T frames per step. For a strictly paper-faithful run, feed a frame-level sampler (shuffle a flat pool of frames, batch 32). Either way, **padded frames are dropped before the forward pass** in the training loop, so BatchNorm statistics come only from real frames.

In [ ]:
class LazyNPYDataset(Dataset):
    """One .npy per utterance. features: [T, F]; labels (optional): [T]."""
    def __init__(self, feature_dir, label_dir=None, feature_dtype=np.float32):
        self.feature_dir = feature_dir
        self.label_dir = label_dir
        self.file_list = sorted(f for f in os.listdir(feature_dir) if f.endswith(".npy"))
        self.feature_dtype = feature_dtype

        if label_dir:
            label_files = sorted(f for f in os.listdir(label_dir) if f.endswith(".npy"))
            assert len(self.file_list) == len(label_files), \
                f"Mismatch: {len(self.file_list)} feature files vs {len(label_files)} label files"
            for f_feat, f_lab in zip(self.file_list, label_files):
                assert os.path.splitext(f_feat)[0] == os.path.splitext(f_lab)[0], \
                    f"Filename mismatch: {f_feat} vs {f_lab}"
            self.label_list = label_files

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        utt_id = os.path.splitext(self.file_list[idx])[0]
        feat_path = os.path.join(self.feature_dir, self.file_list[idx])
        features = np.load(feat_path, allow_pickle=False).astype(self.feature_dtype, copy=False)
        features = torch.as_tensor(features, dtype=torch.float32)

        if self.label_dir:
            label_path = os.path.join(self.label_dir, self.label_list[idx])
            labels = torch.as_tensor(np.load(label_path), dtype=torch.long)
            return features, labels, utt_id
        return features, utt_id


def collate_fn(batch):
    has_labels = len(batch[0]) == 3
    if has_labels:
        feats, labels, utt_ids = zip(*batch)
    else:
        feats, utt_ids = zip(*batch)

    lengths = torch.tensor([f.size(0) for f in feats], dtype=torch.long)
    padded_feats = pad_sequence(feats, batch_first=True)            # [B,T,F]
    B, T, _ = padded_feats.shape
    mask = torch.arange(T).unsqueeze(0) < lengths.unsqueeze(1)      # [B,T] bool

    if has_labels:
        padded_labels = pad_sequence(labels, batch_first=True, padding_value=-100)
        return padded_feats, padded_labels, mask, list(utt_ids)
    return padded_feats, mask, list(utt_ids)


def make_loader(dataset, batch_size=32, shuffle=True):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      pin_memory=True, collate_fn=collate_fn)


def flatten(x_bt_f):
    """[B,T,F] -> [B*T,F]"""
    return x_bt_f.reshape(-1, x_bt_f.size(-1))


def flatten_mask(mask_bt):
    """[B,T] -> [B*T]"""
    return mask_bt.reshape(-1)

## 7) Training the DSN

- Loss weights are Eq. (1): `beta_domain=0.25` (L_sim), `gamma_diff=0.075` (L_diff), `delta_recon=0.1` (L_recon).
- `alpha` (GRL strength) ramps 0→1 via `grl_alpha_schedule`.
- Domain loss is switched on only after `domain_warmup_steps` (10,000, Sec 3.4).
- `scheduler.step()` runs every **mini-batch**, so `StepLR(step_size=20000, gamma=0.95)` decays every 20,000 steps (Sec 3.3).
- Padded frames are dropped (`xf[mask]`) **before** the forward pass, so BatchNorm sees only real frames.
- `recon_type` selects MSE (Eq. 3, default) or SIMSE (Eq. 4) for the Sec 4.1.3 ablation.

In [ ]:
def train_dsn(model, src_loader, tgt_loader, num_epochs=20, lr=0.01,
              beta_domain=0.25, gamma_diff=0.075, delta_recon=0.1,
              domain_warmup_steps=10000, lr_decay_steps=20000, lr_decay_gamma=0.95,
              recon_type='mse', device=device):
    assert recon_type in ('mse', 'simse')
    recon_fn = reconstruction_loss if recon_type == 'mse' else simse_loss

    model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=lr_decay_steps, gamma=lr_decay_gamma)
    ce = nn.CrossEntropyLoss(ignore_index=-100)

    steps_per_epoch = min(len(src_loader), len(tgt_loader))
    total_steps = max(steps_per_epoch * num_epochs, 1)
    global_step = 0

    for epoch in range(1, num_epochs + 1):
        model.train()
        running = 0.0
        alpha = 0.0

        for (src_x, src_y, src_mask, _), (tgt_x, tgt_mask, _) in zip(src_loader, tgt_loader):
            src_x, src_y, src_mask = src_x.to(device), src_y.to(device), src_mask.to(device)
            tgt_x, tgt_mask = tgt_x.to(device), tgt_mask.to(device)

            # drop padding BEFORE the forward pass so BatchNorm sees only real frames
            src_mf, tgt_mf = flatten_mask(src_mask), flatten_mask(tgt_mask)
            src_xf = flatten(src_x)[src_mf]
            src_yf = src_y.reshape(-1)[src_mf]
            tgt_xf = flatten(tgt_x)[tgt_mf]

            alpha = grl_alpha_schedule(global_step / total_steps)

            out_s = model(src_xf, domain='source', mode='train', alpha=alpha)
            out_t = model(tgt_xf, domain='target', mode='train', alpha=alpha)

            l_class = ce(out_s["senone_logits"], src_yf)                     # source-only

            dom_s = torch.zeros(src_xf.size(0), dtype=torch.long, device=device)
            dom_t = torch.ones(tgt_xf.size(0), dtype=torch.long, device=device)
            l_sim = ce(out_s["domain_logits"], dom_s) + ce(out_t["domain_logits"], dom_t)

            l_diff = difference_loss(out_s["private"], out_s["shared"]) + \
                     difference_loss(out_t["private"], out_t["shared"])

            l_recon = recon_fn(src_xf, out_s["recon"]) + recon_fn(tgt_xf, out_t["recon"])

            domain_active = 1.0 if global_step >= domain_warmup_steps else 0.0
            loss = l_class \
                   + beta_domain * domain_active * l_sim \
                   + gamma_diff * l_diff \
                   + delta_recon * l_recon

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            running += loss.item()
            global_step += 1

        print(f"[DSN] epoch {epoch:02d}/{num_epochs} | avg loss {running/max(steps_per_epoch,1):.4f} | alpha {alpha:.3f}")
    return model

## 8) Inference, decoding hooks & sanity metrics

- **`infer_senones`** runs `mode='inference'` (shared encoder → senone classifier only, Sec 3.5) and returns per-utterance senone sequences keyed by utterance ID.
- **`estimate_log_priors` / `prior_normalized_scores`** implement Sec 3.5's "pre-softmax output normalized using log probability of priors": subtract the log senone priors from the log-softmax posteriors to get the pseudo-log-likelihoods a Kaldi HCLG (WFST) decoder consumes.
- **`frame_level_accuracy`** is an honest sanity metric (needs a labeled loader) — not the paper's WER.
- **`domain_accuracy`** reports the fraction of frames the shared-encoder's domain classifier labels correctly (Table 2 quantifies domain-invariance this way).

> **True WER (Table 1) is out of scope here.** It requires decoding the prior-normalized senone posteriors through a Kaldi HMM/WFST pipeline (HCLG from the Sanskrit lexicon L-FST via the SLP1-M G2P scheme + a bi-gram grammar G-FST, Sec 3.5). `prior_normalized_scores` produces exactly the frame scores that pipeline expects.

In [ ]:
@torch.no_grad()
def infer_senones(model, loader, device=device):
    """Returns {utt_id: predicted_senone_sequence (np.ndarray)}."""
    model.eval()
    predictions = {}
    for feats, mask, utt_ids in loader:
        feats, mask = feats.to(device), mask.to(device)
        xf, mf = flatten(feats), flatten_mask(mask)
        out = model(xf[mf], mode='inference')
        preds = out["senone_logits"].argmax(dim=1).cpu()

        lengths = mask.sum(dim=1).cpu().tolist()
        start = 0
        for uid, length in zip(utt_ids, lengths):
            predictions[uid] = preds[start:start + length].numpy()
            start += length
    return predictions


@torch.no_grad()
def estimate_log_priors(labeled_loader, num_senones=3080, eps=1e-8, device=device):
    """Log senone priors from training-label frequencies (Sec 3.5)."""
    counts = torch.zeros(num_senones, dtype=torch.double)
    for _, labels, mask, _ in labeled_loader:
        yf = labels.reshape(-1)[flatten_mask(mask)]
        yf = yf[yf >= 0]
        counts += torch.bincount(yf, minlength=num_senones).double()
    priors = counts / counts.sum().clamp_min(1)
    return torch.log(priors + eps).to(device)


@torch.no_grad()
def prior_normalized_scores(model, feats, log_priors, device=device):
    """log P(senone|x) - log P(senone): pseudo-log-likelihoods for WFST decoding."""
    model.eval()
    logits = model(feats.to(device), mode='inference')["senone_logits"]
    return F.log_softmax(logits, dim=1) - log_priors


@torch.no_grad()
def frame_level_accuracy(model, labeled_loader, device=device):
    """Sanity metric only (NOT the paper's WER) -- needs a labeled loader."""
    model.eval()
    correct, total = 0, 0
    for feats, labels, mask, _ in labeled_loader:
        feats, labels, mask = feats.to(device), labels.to(device), mask.to(device)
        xf, yf, mf = flatten(feats), labels.reshape(-1), flatten_mask(mask)
        out = model(xf[mf], mode='inference')
        preds = out["senone_logits"].argmax(dim=1)
        correct += (preds == yf[mf]).sum().item()
        total += int(mf.sum().item())
    return correct / max(total, 1)


@torch.no_grad()
def domain_accuracy(model, loader, domain_label, device=device):
    """Fraction of frames labeled with the correct domain (0=source, 1=target).
    Pass the source (Hindi-dev) loader with domain_label=0 and the target
    (Sanskrit-test) loader with domain_label=1. Lower = more domain-invariant."""
    model.eval()
    correct, total = 0, 0
    for batch in loader:
        feats, mask = batch[0], batch[-2]
        feats, mask = feats.to(device), mask.to(device)
        xf = flatten(feats)[flatten_mask(mask)]
        logits = model.domain_classifier(model.shared_encoder(xf))
        preds = logits.argmax(dim=1)
        correct += (preds == domain_label).sum().item()
        total += preds.numel()
    return correct / max(total, 1)

## 9) t-SNE of shared features (Sec 4.1.2)

Collects shared-encoder outputs for frames of a chosen senone from the source and target loaders and plots a 2-D t-SNE (Fig. 4). Intermingled H/S points indicate domain-invariant shared features.

In [ ]:
@torch.no_grad()
def collect_shared_feats(model, loader, senone_id=None, max_points=2000, device=device):
    """Returns shared-encoder features [N, D]. If senone_id is given, keeps only
    frames with that label (requires a labeled loader)."""
    model.eval()
    feats_out = []
    for batch in loader:
        if len(batch) == 4:      # labeled: feats, labels, mask, ids
            feats, labels, mask, _ = batch
        else:                    # unlabeled: feats, mask, ids
            feats, mask = batch[0], batch[1]
            labels = None
        feats, mask = feats.to(device), mask.to(device)
        mf = flatten_mask(mask)
        xf = flatten(feats)[mf]
        if senone_id is not None and labels is not None:
            yf = labels.reshape(-1)[mf].to(device)
            xf = xf[yf == senone_id]
        if xf.size(0):
            feats_out.append(model.shared_encoder(xf).cpu())
        if sum(f.size(0) for f in feats_out) >= max_points:
            break
    return torch.cat(feats_out)[:max_points] if feats_out else torch.empty(0)


def plot_tsne(model, src_loader, tgt_loader, senone_id=None, max_points=2000, device=device):
    from sklearn.manifold import TSNE
    import matplotlib.pyplot as plt

    src = collect_shared_feats(model, src_loader, senone_id, max_points, device).numpy()
    tgt = collect_shared_feats(model, tgt_loader, senone_id, max_points, device).numpy()
    if len(src) == 0 or len(tgt) == 0:
        print("Not enough frames to plot (check senone_id / loaders).")
        return

    X = np.concatenate([src, tgt], axis=0)
    emb = TSNE(n_components=2, init="pca", perplexity=30).fit_transform(X)
    ns = len(src)
    plt.figure(figsize=(6, 6))
    plt.scatter(emb[:ns, 0], emb[:ns, 1], s=6, label="H (source)", alpha=0.6)
    plt.scatter(emb[ns:, 0], emb[ns:, 1], s=6, label="S (target)", alpha=0.6)
    plt.legend(); plt.title("t-SNE of shared features"
                            + (f" (senone {senone_id})" if senone_id is not None else ""))
    plt.tight_layout(); plt.show()

## 10) Example usage — wired to the prepared dataset

Data prepared with the Kaldi pipeline in `prepare_data/` (m4a→8 kHz wav → Kaldi HMM-GMM
alignment → 1320-d fbank features → float16 `.npy`). Layout under `D:\uda_prep\npy`:

- `hindi\train` + `hindi\train_lab` — source features + senone labels (14,990 utts; 10 dropped for failed alignment)
- `sanskrit\train` — target adaptation features (2,837, unlabeled)
- `sanskrit\test`  — target test features (558, unlabeled)

**`num_senones = 2472`** — the pdf-id count from our Kaldi tri3 tree (grapheme lexicon).
The paper reports 3080 with a phonemic Hindi G2P; the DSN is agnostic to the exact count.

> **Compute note:** on CPU this is impractical for the full 14,990-utt × 20-epoch run — use a
> CUDA build of PyTorch and a GPU. `.npy` are stored float16 and cast to float32 at load.

In [ ]:
NPY = r"D:\uda_prep\npy"
NUM_SENONES = 2472
batch_size = 32

src_loader      = make_loader(LazyNPYDataset(NPY + r"\hindi\train", NPY + r"\hindi\train_lab"), batch_size, shuffle=True)
tgt_loader      = make_loader(LazyNPYDataset(NPY + r"\sanskrit\train"),                          batch_size, shuffle=True)
tgt_test_loader = make_loader(LazyNPYDataset(NPY + r"\sanskrit\test"),                           batch_size, shuffle=False)

# --- train ----------------------------------------------------------------
dsn = DSN(input_dim=1320, shared_dim=1024, private_dim=512, num_senones=NUM_SENONES)
dsn = train_dsn(dsn, src_loader, tgt_loader, num_epochs=20)                    # MSE recon (Eq. 3)
# dsn = train_dsn(dsn, src_loader, tgt_loader, recon_type='simse')            # SIMSE variant (Sec 4.1.3)
torch.save(dsn.state_dict(), "dsn_model.pth")

# --- inference / decode hooks ---------------------------------------------
log_priors = estimate_log_priors(src_loader, num_senones=NUM_SENONES)          # Sec 3.5 priors
preds = infer_senones(dsn, tgt_test_loader)                                    # {utt_id: senone seq}
# domain-invariance check (Table 2 style): needs a target-labeled or source loader
# print("domain acc S-test:", domain_accuracy(dsn, tgt_test_loader, 1))
# feed prior_normalized_scores(dsn, frames, log_priors) into a Kaldi HCLG decoder for WER.

## 11) Still external / out of scope

- **Kaldi WFST/HCLG decoding for word-level WER** (Sec 3.5): needs Kaldi, the SLP1-M Sanskrit pronunciation dictionary (L-FST), and a bi-gram LM (G-FST). This notebook produces the prior-normalized senone scores that feed it (`prior_normalized_scores`); the decoder itself is a separate toolchain.
- **HMM-GMM senone alignments** (Sec 3.1): the per-frame senone labels come from a Kaldi HMM-GMM system; assumed precomputed into the label `.npy` files.